In [1]:
import os
import re
import pdfplumber

In [2]:
DATA_PATH = os.path.join("samples")
pdf_paths = []
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dir not found: {DATA_PATH}")

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        pdf_paths.append(os.path.join(DATA_PATH, file))

print(f"Znaleziono pliki: {pdf_paths}")


Znaleziono pliki: ['samples\\OC_os_fiz_przy_EDU_Plus_2b489658.pdf']


In [3]:
pdfs_texts = []

for pdf_path in pdf_paths:
    full_pdf_text = ""

    print(f"Text extraction from: {os.path.basename(pdf_path)}\n")
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            temp_text = page.extract_text(layout=True)
            full_pdf_text += temp_text + "\n"
    pdfs_texts.append(" ".join(full_pdf_text.split()))
    print(f"Extracted text from {os.path.basename(pdf_path)}: {len(full_pdf_text)} characters\n")

print("Check of first 300 characters of the first PDF text:\n")
preview = pdfs_texts[0][:300] if pdfs_texts else ""
print(preview + ("\n[...]" if len(pdfs_texts[0]) > 300 else ""))


Text extraction from: OC_os_fiz_przy_EDU_Plus_2b489658.pdf

Extracted text from OC_os_fiz_przy_EDU_Plus_2b489658.pdf: 156246 characters

Check of first 300 characters of the first PDF text:

Ubezpieczenie Odpowiedzialności cywilnej osób fizycznych w życiu prywatnym oraz nauczycieli i dyrektorów placówek oświatowych w ramach oferty EDU Plus Dokument zawierający informacje o produkcie ubezpieczeniowym Przedsiębiorstwo: InterRisk Towarzystwo Ubezpieczeń Spółka Akcyjna Vienna Insurance Grou
[...]


In [ ]:
import re
pdf_tokens = re.findall(
    r"\d{2}-\d{3}|[\w]+|[^\w\s]",
    "al. Wojska Polskiego 496/58 ulica sojkowa os. Wojskowe 61-245 07.07.2005 " + pdfs_texts[0] if pdfs_texts else "",
    flags=re.UNICODE
)
pdf_tokens

['ulica',
 'sojkowa',
 'os',
 '.',
 'Wojskowe',
 '61-245',
 '07',
 '.',
 '07',
 '.',
 '2005',
 'Ubezpieczenie',
 'Odpowiedzialności',
 'cywilnej',
 'osób',
 'fizycznych',
 'w',
 'życiu',
 'prywatnym',
 'oraz',
 'nauczycieli',
 'i',
 'dyrektorów',
 'placówek',
 'oświatowych',
 'w',
 'ramach',
 'oferty',
 'EDU',
 'Plus',
 'Dokument',
 'zawierający',
 'informacje',
 'o',
 'produkcie',
 'ubezpieczeniowym',
 'Przedsiębiorstwo',
 ':',
 'InterRisk',
 'Towarzystwo',
 'Ubezpieczeń',
 'Spółka',
 'Akcyjna',
 'Vienna',
 'Insurance',
 'Group',
 'z',
 'siedzibą',
 'w',
 'Polsce',
 ',',
 'ul',
 '.',
 'Noakowskiego',
 '22',
 ',',
 '00-668',
 'Warszawa',
 ',',
 'numer',
 'zezwolenia',
 'Ministra',
 'Finansów',
 'DU',
 '/',
 '905',
 '/',
 'A',
 '/',
 'KP',
 '/',
 '93',
 'z',
 '5',
 'listopada',
 '1993',
 'roku',
 'Produkt',
 ':',
 'Odpowiedzialność',
 'cywilna',
 'osób',
 'fizycznych',
 'w',
 'życiu',
 'prywatnym',
 'oraz',
 'nauczycieli',
 'i',
 'dyrektorów',
 'placówek',
 'oświatowych',
 'w',
 'ramach

In [ ]:
'''
    Requirements state that one token should be treated as one word, 
    and the tokenizer should be able to handle special characters and punctuation. 
    The tokenizer will be used to split the text into manageable chunks for further processing.
'''
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "allegro/herbert-base-cased", use_fast=True
)

model_max_length = tokenizer.model_max_length
if model_max_length is None or model_max_length > 100_000:
    model_max_length = 512

special_tokens_count = tokenizer.num_special_tokens_to_add(pair=False)
TOKENIZER_CHUNK_SIZE = max(1, model_max_length - special_tokens_count)

print(f"Tokenizer gotowy. Limit treści chunku: {TOKENIZER_CHUNK_SIZE} tokenów.")


Pobieranie i ładowanie tokenizera: allegro/herbert-base-cased...
Tokenizer gotowy. Limit treści chunku: 510 tokenów.


In [ ]:
def split_text_into_chunks(text, tokenizer, max_tokens):
    parts = re.findall(r"\s*\S+", text)
    chunks = []
    current_parts = []
    current_token_count = 0

    for part in parts:
        part_token_count = len(tokenizer.tokenize(part))

        if current_parts and current_token_count + part_token_count > max_tokens:
            chunks.append("".join(current_parts))
            current_parts = [part]
            current_token_count = part_token_count
        else:
            current_parts.append(part)
            current_token_count += part_token_count

    if current_parts:
        chunks.append("".join(current_parts))

    return chunks

chunks = split_text_into_chunks(full_pdf_text, tokenizer, TOKENIZER_CHUNK_SIZE)
chunk_token_counts = [len(tokenizer.tokenize(chunk)) for chunk in chunks]

print(f"Utworzono {len(chunks)} chunków wyłącznie na potrzeby limitu tokenizera.")
print(f"Najdłuższy chunk: {max(chunk_token_counts, default=0)} / {TOKENIZER_CHUNK_SIZE} tokenów.")


Utworzono 53 chunków wyłącznie na potrzeby limitu tokenizera.
Najdłuższy chunk: 510 / 510 tokenów.


In [20]:
tokens = []

for chunk in chunks:
    tokens.extend(tokenizer.tokenize(chunk))

print(f"Tokeny całego PDF-a: {len(tokens)}")
print(tokens[:50])


Tokeny całego PDF-a: 23782
['Ubezpieczenie</w>', 'Odpowiedzi', 'alności</w>', 'cywilnej</w>', 'osób</w>', 'fizycznych</w>', 'w</w>', 'życiu</w>', 'prywatnym</w>', 'oraz</w>', 'nauczycieli</w>', 'i</w>', 'dyrektorów</w>', 'placówek</w>', 'oświatowych</w>', 'w</w>', 'ramach</w>', 'oferty</w>', 'E', 'D', 'U</w>', 'Plus</w>', 'Dokument</w>', 'zawierający</w>', 'informacje</w>', 'o</w>', 'produ', 'kcie</w>', 'ubezpieczeniowym</w>', 'Przedsiębiorstwo</w>', ':</w>', 'Inter', 'Ri', 'sk</w>', 'Towarzystwo</w>', 'Ubezpieczeń</w>', 'Spółka</w>', 'Ak', 'cyjna</w>', 'V', 'ien', 'na</w>', 'In', 'surance</w>', 'Group</w>', 'z</w>', 'siedzibą</w>', 'w</w>', 'Polsce</w>', ',</w>']


In [21]:
tokenization_flow = {
    "Full_pdf_text": Full_pdf_text,
    "chunks": chunks,
    "tokens": tokens,
}

print("Gotowy przepływ:")
print(f"Full_pdf_text: {len(tokenization_flow['Full_pdf_text'])} znaków")
print(f"chunks: {len(tokenization_flow['chunks'])} elementów")
print(f"tokens: {len(tokenization_flow['tokens'])} elementów")


Gotowy przepływ:
Full_pdf_text: 155297 znaków
chunks: 53 elementów
tokens: 23782 elementów


In [22]:
assert isinstance(Full_pdf_text, str)
assert isinstance(chunks, list)
assert isinstance(tokens, list)
assert all(isinstance(chunk, str) for chunk in chunks)
assert all(isinstance(token, str) for token in tokens)
assert sum(chunk_token_counts) == len(tokens)

print("Walidacja struktury OK: Full_pdf_text -> chunks -> tokens")


Walidacja struktury OK: Full_pdf_text -> chunks -> tokens
